# 08 · Human-in-the-Loop: Pausing a Flow for a Human

> **Kernel:** `Phase 5 · CrewAI` · **Needs:** Ollama.

Some steps need a person's decision: approving a post, a refund, a published
report. CrewAI Flows support this with the **`@human_feedback`** decorator: after
a step runs, the flow asks a human, turns their free-text answer into an
**outcome** (like `approved` / `rejected`), and routes on it. With a
**non-blocking provider** and **persistence**, the flow pauses, your process
exits, and the flow resumes later, even in a different process.

**You will learn**
- `@human_feedback`: message, outcomes (`emit`), and how feedback becomes an outcome
- Why the default provider **blocks**, and how to pause instead (`HumanFeedbackPending`)
- Resuming with `Flow.from_pending(...)` and `resume_async(feedback)`, in a **new process**
- Reading the feedback: `last_human_feedback` (and a documentation trap)
- What an LLM does with **ambiguous** feedback, and how to avoid surprises

**Prerequisites:** notebook 06 (Flows).

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')                       # hide noisy library warnings in the notebook
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 03_CREWAI folder
sys.path.insert(0, str(SECTION))
from common import get_llm, quiet_console              # import common BEFORE crewai (telemetry off)
quiet_console()                                         # hide CrewAI's decorative console panels

import json, subprocess, sys, textwrap, pathlib
from pydantic import BaseModel
from crewai.flow import Flow, start, listen, human_feedback, HumanFeedbackPending
from crewai.flow.async_feedback import HumanFeedbackProvider
from crewai.flow.persistence import SQLiteFlowPersistence

llm = get_llm()
WORK = pathlib.Path('checkpoints')
WORK.mkdir(exist_ok=True)

## 1 · Why: a pause, not a blocked process

Asking a human with `input()` blocks the program until someone types. Real
approvals take hours, and the process will have been restarted by then. What you
want is:

```
 flow runs ─► step "write" ─► @human_feedback: ask the reviewer
                                 │ provider raises HumanFeedbackPending
                                 ▼
            state + "waiting for feedback on write" saved to SQLite
            kickoff returns a HumanFeedbackPending object   ← nobody is blocked
                         … hours later, any process …
 Flow.from_pending(flow_id) ─► resume_async("Looks good, ship it!")
            an LLM maps the text to an outcome: "approved"
            ─► @listen("approved") runs
```

## 2 · `@human_feedback` on a flow step

Put `@human_feedback` under the step's `@start`/`@listen`. After the step
finishes, its return value is shown to the human with your `message`. `emit`
lists the possible **outcomes**, and `llm` is the model that maps the human's
free text onto one of them. Other steps `@listen` for those outcomes.

By default feedback is collected by a **console provider** that calls `input()`
and blocks. That's fine in a terminal script, and useless for real approvals. We
write a tiny provider that **pauses** instead: it raises `HumanFeedbackPending`,
which tells the flow to save itself and return.

In [ ]:
class PauseForReviewer(HumanFeedbackProvider):
    # Non-blocking: in a real app this would post to Slack, email, or a review queue.
    def request_feedback(self, context, flow):
        print(f'  📨 review requested for step {context.method_name!r}: {str(context.method_output)[:70]}…')
        raise HumanFeedbackPending(context=context)


class Post(BaseModel):
    topic: str = ''
    draft: str = ''
    status: str = ''


class PublishFlow(Flow[Post]):
    @start()
    @human_feedback(message='Publish this post?', emit=['approved', 'rejected'],
                    llm=llm, provider=PauseForReviewer())
    def write(self):
        self.state.draft = llm.call(f'One-sentence LinkedIn post about {self.state.topic}. Sentence only.')
        return self.state.draft

    @listen('approved')
    def publish(self):
        self.state.status = f'published (reviewer said: {self.last_human_feedback.feedback!r})'
        return self.state.status

    @listen('rejected')
    def discard(self):
        self.state.status = f'discarded (reviewer said: {self.last_human_feedback.feedback!r})'
        return self.state.status

**Persistence is required to pause.** The paused flow must be saved somewhere so
another process can find it. `SQLiteFlowPersistence(path)` stores the state and
the pending feedback request.

In [ ]:
DB = WORK / 'review_flows.db'
DB.unlink(missing_ok=True)
store = SQLiteFlowPersistence(str(DB))

flow = PublishFlow(persistence=store, suppress_flow_events=True)
result = await flow.kickoff_async(inputs={'topic': 'robots that handle rain'})
print('kickoff returned:', type(result).__name__)
print('flow id         :', flow.state.id, '← the handle for resuming')

`kickoff_async` **returned**, with a `HumanFeedbackPending` result instead of a
final value. (CrewAI still prints a "Flow Paused" panel here; it's informational.)
Nothing is blocked; the flow is saved under its id.

## 3 · Resuming, possibly much later

Any code that knows the flow id can load the paused flow with
`Flow.from_pending(flow_id, persistence)`, inspect what's pending, and resume
with the human's words:

In [ ]:
paused = PublishFlow.from_pending(flow.state.id, store, suppress_flow_events=True)
print('waiting on :', paused.pending_feedback.method_name)
print('question   :', paused.pending_feedback.message)
print('shown      :', paused.pending_feedback.method_output)

final = await paused.resume_async('Looks good, ship it!')
print('\noutcome    :', paused.last_human_feedback.outcome)
print('result     :', final)

The reviewer wrote free text; the LLM mapped *"Looks good, ship it!"* to the
outcome `approved`, and the `publish` step ran.

**A documentation trap.** CrewAI's own docstring examples read the feedback as
`self.human_feedback`. In this version that attribute doesn't exist, and the
flow crashes with `AttributeError`. The working attribute is
**`self.last_human_feedback`** (a `HumanFeedbackResult` with `output`, `feedback`,
`outcome`, `timestamp`), and `self.human_feedback_history` keeps all of them.
When docs and behaviour disagree, trust a quick test.

## 4 · Across processes: the real use case

Now the full story with separate processes: one process starts the flow and
exits while it's paused; later, a completely **new process** loads it by id and
resumes it with the reviewer's decision.

In [ ]:
APP = WORK / 'review_app.py'
APP.write_text(textwrap.dedent(f"""
    import asyncio, sys
    sys.path.insert(0, {str(SECTION)!r})
    from common import get_llm
    from pydantic import BaseModel
    from crewai.flow import Flow, start, listen, human_feedback, HumanFeedbackPending
    from crewai.flow.async_feedback import HumanFeedbackProvider
    from crewai.flow.persistence import SQLiteFlowPersistence

    llm = get_llm()
    store = SQLiteFlowPersistence('refunds.db')

    class Pause(HumanFeedbackProvider):
        def request_feedback(self, context, flow):
            raise HumanFeedbackPending(context=context)

    class Refund(BaseModel):
        order: str = ''
        amount: int = 0
        status: str = ''

    class RefundFlow(Flow[Refund]):
        @start()
        @human_feedback(message='Approve this refund?', emit=['approved', 'rejected'], llm=llm, provider=Pause())
        def propose(self):
            return f'Refund {{self.state.amount}} EUR for order {{self.state.order}}'

        @listen('approved')
        def pay(self):
            with open('payments.log', 'a') as f:
                f.write(f'{{self.state.order}},{{self.state.amount}}\\n')
            self.state.status = 'refund paid'
            return self.state.status

        @listen('rejected')
        def decline(self):
            self.state.status = 'refund declined'
            return self.state.status

    async def main(cmd, arg):
        if cmd == 'start':
            flow = RefundFlow(persistence=store, suppress_flow_events=True)
            await flow.kickoff_async(inputs={{'order': 'B-52', 'amount': 180}})
            print('PAUSED', flow.state.id)
        else:
            flow_id, feedback = arg.split('|', 1)
            flow = RefundFlow.from_pending(flow_id, store, suppress_flow_events=True)
            result = await flow.resume_async(feedback)
            print('RESUMED', flow.last_human_feedback.outcome, '->', result)

    asyncio.run(main(sys.argv[1], sys.argv[2] if len(sys.argv) > 2 else ''))
"""))

def app(*args):
    out = subprocess.run([sys.executable, APP.name, *args], cwd=WORK, capture_output=True, text=True)
    lines = [l for l in out.stdout.splitlines() if l.startswith(('PAUSED', 'RESUMED'))]
    print(f'$ python review_app.py {" ".join(args)[:40]}\n  ' + ('\n  '.join(lines) or out.stderr[-300:]))
    return lines

for f in ('refunds.db', 'payments.log'):
    (WORK / f).unlink(missing_ok=True)
flow_id = app('start')[0].split()[1]
print('payments.log exists?', (WORK / 'payments.log').exists())

In [ ]:
app('resume', f'{flow_id}|Approved, the order really arrived broken.')     # a NEW process
print('payments.log:', (WORK / 'payments.log').read_text().strip())

The first process exited while the refund waited. The second knew only the flow
id and the database, and it finished the flow, paying the refund exactly once.

## 5 · Ambiguous feedback: who decides?

The mapping from free text to an outcome is done by an **LLM**. That's
convenient ("ship it!", "LGTM", "yes please" all mean approved), but what does
it do with an unclear answer?

In [ ]:
for feedback in ['approved', 'No, the tone is wrong.', 'hmm, maybe later?']:
    f = PublishFlow(persistence=store, suppress_flow_events=True)
    await f.kickoff_async(inputs={'topic': 'robot safety'})
    paused = PublishFlow.from_pending(f.state.id, store, suppress_flow_events=True)
    await paused.resume_async(feedback)
    print(f'{feedback!r:<28} → outcome = {paused.last_human_feedback.outcome}')

In our runs, *"hmm, maybe later?"* was mapped to **rejected**. That's not
unreasonable, but the model made that decision, not the reviewer. For approvals
that matter:

- **Offer exact choices** in the UI (buttons that send `approved` / `rejected`),
  and use free text only as a comment.
- Consider a **third outcome** (`emit=['approved', 'rejected', 'needs_info']`) so
  uncertainty has somewhere to go.
- `default_outcome=` sets what an empty response means.

**Crew tasks can also ask a human.** `Task(human_input=True)` makes the agent ask
for feedback on its answer before finishing, but it **blocks** on console input,
so it suits interactive scripts, not services. For anything that waits on a real
person, use a Flow with `@human_feedback` and a non-blocking provider.

## 6 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Default console provider in a service | the process hangs waiting for `input()` | a provider that raises `HumanFeedbackPending` |
| No persistence | nothing to resume from | `SQLiteFlowPersistence` (or another backend) |
| Losing the flow id | the paused flow can't be found | store `state.id` with the request you send to the reviewer |
| `self.human_feedback` (from the docs) | `AttributeError` | `self.last_human_feedback` |
| Free-text approvals for high-stakes actions | the LLM decides ambiguous cases | exact choices; a "needs info" outcome |
| `human_input=True` in a web service | a blocked worker | Flows with `@human_feedback` |

## 7 · Check yourself

<details><summary><b>1.</b> What does <code>emit=['approved','rejected']</code> plus <code>llm=...</code> do?</summary>

It defines the possible outcomes, and uses the LLM to map the human's free-text feedback onto one of them. Steps listening for that outcome run next.
</details>

<details><summary><b>2.</b> How does a flow pause without blocking?</summary>

A feedback provider raises `HumanFeedbackPending`. With persistence configured, the flow saves its state and the pending request, and `kickoff` returns a `HumanFeedbackPending` object instead of waiting.
</details>

<details><summary><b>3.</b> What does a different process need in order to resume the flow?</summary>

The flow class, the same persistence backend, and the flow id: `Flow.from_pending(flow_id, persistence)`, then `await flow.resume_async(feedback)`.
</details>

<details><summary><b>4.</b> A reviewer writes "maybe later". What might happen, and how do you design around it?</summary>

The LLM may map it to an outcome the reviewer didn't intend (we saw "rejected"). Offer exact choices, add a "needs info" outcome, or set `default_outcome`.
</details>

## Takeaway

`@human_feedback` turns a flow step into a review point: the step's output goes to
a human, their words become an outcome, and the flow routes on it. With a
non-blocking provider and persistence, the flow **pauses and resumes from any
process** by id. Read feedback from `last_human_feedback`, and don't let an LLM
settle ambiguous approvals for you.

Next → `09_memory_and_persistence.ipynb`: remembering across runs, and what
survives a crash.